# TFM: Análisis de Políticas de Sostenibilidad mediante técnicas de Argumentacion Computacional

## Detección de Argumentos con deepseek-r1:70b en poliGPT API

- ollama serve
- ollama run gemma3:4b

In [1]:
%pip install langchain pymupdf openai openpyxl pandas numpy openpyxl --quiet

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [33]:
from typing import List
from pydantic import BaseModel, Field, ValidationError
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate
from langchain_core.exceptions import OutputParserException
import requests
import json
import re
from openai import OpenAI
import openai
import httpx
import pandas as pd
import numpy as np
import os
import openpyxl

process_text_path = "..\\Data\\Processed Files (sections)\\"

model_name="deepseek-r1:70b"
prefix = 'OCDE_2024'
output_dir = "..\\Data\\Extracted Arguments No Keywords (all text)\\"

## Input text processing

In [34]:
# 1. Define your Pydantic schema for output
class ArgumentResponse(BaseModel):
    arguments: List[str] = Field(..., description="List of arguments extracted directly from the text.")

# 2. Setup output parser
pydantic_parser = PydanticOutputParser(pydantic_object=ArgumentResponse)

# 3. Extend text with first sentence from the next page
def extend_pages_with_next_sentence(pages):
    def get_first_sentence(text):
        match = re.search(r'(.+?\.)', text.strip())
        return match.group(1).strip() if match else ""

    extended_pages = []
    for i, page in enumerate(pages):
        current_text = page["text"]
        if i + 1 < len(pages):
            next_sentence = get_first_sentence(pages[i + 1]["text"])
            current_text += " " + next_sentence
        extended_pages.append({
            "page": page["page"],
            "text": current_text
        })
    return extended_pages

# 4. Build the prompt and call the LLM to extract arguments
def extract_arguments_json(text, topic, model_name) -> ArgumentResponse:
    format_instructions = pydantic_parser.get_format_instructions()

    prompt = PromptTemplate(
        template=(
            "Task: Text Span Identification for Arguments related to Sustainable Development Goal: {topic}\n\n"
            "Role: You are an expert in logical reasoning, sustainability reporting, and argument analysis. "
            "Your job is to identify and extract **verbatim arguments** about {topic} from long-form sustainability texts.\n\n"
            "Instructions:\n"
            "1. Carefully read the entire input text.\n"
            "2. Identify ONLY those sentences or phrases that:\n"
            "   - Clearly support or argue for or against the topic {topic}\n"
            "   - Contain keyword from the relevant lists below\n"
            "   - Are exclusively about {topic} (EXCLUDE if they mention or refer to other SDGs or unrelated sustainability topics)\n\n"
            "3. Each extracted argument must:\n"
            "   - Relate exclusively to the specified SDG ({topic})\n"
            "   - Stand as a full statement\n"
            "   - Be copied exactly from the original (no paraphrasing)\n"
            "   - Include only the necessary context for understanding\n"
            "4. If no qualifying arguments are found, return an empty array.\n\n"
            "Output Rules:\n"
            "- Use **only the exact text** from the original\n"
            "- Do **not** add or reword anything\n"
            "- Return only valid JSON\n"
            "- No markdown (```), no extra explanation\n\n"
            "Text:\n\"\"\"\n{text}\n\"\"\"\n\n"
            "Respond ONLY with a JSON object like this:\n\n"
            "{format_instructions}"
        ),
        input_variables=["text", "topic"],
        partial_variables={"format_instructions": format_instructions}
    )

    final_prompt = prompt.format_prompt(text=text, topic=topic).to_string()

    client = OpenAI(
    base_url = 'https://api.poligpt.upv.es',  
    api_key = 'sk-Icbf-5FyeV0QcLWBC9SNEA',
    timeout=180                    # seconds (global)     
        )

    chat_completion = client.chat.completions.create(
        messages = [
            {'role': 'system', 'content': 'You are an expert in logical reasoning, sustainability reporting, and argument analysis.'},
            {'role': 'user', 'content': final_prompt}
        ],
        model = model_name,
        temperature = 0,
    )

    raw_output = chat_completion.choices[0].message.content
    
    try:
        return pydantic_parser.parse(raw_output)
    except OutputParserException as err:
        print("Parse failed:", err)
        return ArgumentResponse(arguments=[])

# 5. Wrapper function for pipeline
def extract_arguments_from_text(text, topic, model_name) -> List[str]:
    result = extract_arguments_json(text, topic, model_name)
    return result.arguments

# 6. Main document-level processor
def process_document(pages, model_name, topic=""):
    extended_pages = extend_pages_with_next_sentence(pages)
    processed = []
    for page in extended_pages:
        print(f"\n--- Processing Page {page['page']} ---")
        #print("Text to analyze:\n", page["text"])
        
        arguments = extract_arguments_from_text(page["text"], topic, model_name)
        
        print("Extracted Arguments:")
        for i, arg in enumerate(arguments, 1):
            print(f"{i}. {arg}")

        processed.append({
            "page": page["page"],
            "text": page["text"],
            "arguments": arguments
        })
    return processed


# 7. File I/O
def save_to_json(processed, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(processed, f, indent=2, ensure_ascii=False)

def process_directory(input_dir, output_dir, prefix, model_name, topic="", sgd_number=None):
    os.makedirs(output_dir, exist_ok=True)
    all_results = []

    for filename in os.listdir(input_dir):
        if filename.endswith(".json") and filename.startswith(prefix):
            filepath = os.path.join(input_dir, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                pages = json.load(f)

            section_name = filename.replace(".json", "")
            processed = process_document(pages, model_name, topic)

            for item in processed:
                item["section"] = section_name  # Add section identifier
                all_results.append(item)
                
    return all_results



## SGD 1: Poverty

In [35]:
topic = "SGD 1 (Poverty): End poverty in all its forms everywhere"
sgd_number = "1"
resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:
1. Progress towards 48% of the SDG targets is currently insufficient, and 37% are either stagnating or regressing, including on key targets such as those related to poverty, hunger and climate action.
2. More specifically, rises in price levels and energy costs, alongside disruptions in global food markets, have adversely affected SDG targets related to poverty and inequality (SDGs 1 and 10), affordable energy (SDG 7) and food security (SDG 2).
3. Our survey of over 175 local and regional governments (LRGs) revealed a decline in living standards due to inflationary pressures and the repercussions of recent shocks among 80% of respondents.

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:
1. In the aftermath of the COVID-19 pandemic and Russia’s war of aggression against Ukraine, the survey puts the spotlight on the declining standards of living in many cities and regions (SDGs 1 and 10).
2. A

## SGD 2: Hunger

In [36]:
topic = "SGD 2 (Hunger): End hunger, achieve food security and improved nutrition and promote sustainable agriculture"
sgd_number = "2"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:
1. More specifically, rises in price levels and energy costs, alongside disruptions in global food markets, have adversely affected SDG targets related to poverty and inequality (SDGs 1 and 10), affordable energy (SDG 7) and food security (SDG 2).
2. around half noted the growing importance of combating hunger (SDG 2).
3. promoting sustainable food systems and reducing food waste.

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:
1. the growing importance of food security (SDG 2)

--- Processing Page 11 ---
Extracted Arguments:
1. LRGs have implemented a variety of measures to address SDG 2 Zero hunger in light of increasingly frequent disruptions in the global food supply chain, notably setting up food banks and food distribution mechanisms for vulnerable populations (46%).
2. The most common action of LRGs to achieve sustainable food systems is the promotion of local food production (60%) w

## SGD 3: Health

In [37]:
topic = "SGD 3 (Health): Ensure healthy lives and promote well-being for all at all ages"
sgd_number = "3"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:
1. The SDGs served as a key framework to guide cities and regions in recovering from the COVID-19 crisis
2. Around two thirds of LRGs are using the SDGs to guide policy making, which represents an increase of nearly 25 percentage points compared to before the pandemic (39%).
3. Almost half (45%) of LRGs consider the People dimension (SDGs 1 to 5) to be the most important post-COVID-19 challenge, which encompasses the SDGs on poverty, food, health, education and gender. cost of living.

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 12 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:

--- Processing Page 1

## SGD 4: Education

In [38]:
topic = "SGD 4 (Education): Ensure inclusive and equitable quality education"
sgd_number = "4"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:
1. the People dimension (SDGs 1 to 5) to be the most important post-COVID-19 challenge, which encompasses the SDGs on poverty, food, health, education and gender.

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 12 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:
1. almost 60% of them prioritised the efficient delivery of social and community services for disadvantaged groups and equitable access to education to reduce inequalities.

--- Processing Page 19 ---
Extracted Arguments:

--- Processing Page 20 ---
Extracted Arguments:

--- Processing Page 21 ---
Extracted Arguments:

--- Processing Page 22 ---
E

## SGD 5: Gender

In [39]:
topic = "SGD 5 (Gender): Achieve gender equality and empower all women and girls"
sgd_number = "5"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:
1. Almost half (45%) of LRGs consider the People dimension (SDGs 1 to 5) to be the most important post-COVID-19 challenge, which encompasses the SDGs on poverty, food, health, education and gender.

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 12 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:

--- Processing Page 19 ---
Extracted Arguments:

--- Processing Page 20 ---
Extracted Arguments:

--- Processing Page 21 ---
Extracted Arguments:

--- Processing Page 22 ---
Extracted Arguments:

--- Processing Page 24 ---
Extracted Arguments:

--- Processing Page 25 ---
Extracted Arguments:

--- Processing Page

## SGD 6: Water and sanitation

In [40]:
topic = "SGD 6 (Water and sanitation): Ensure availability and sustainable management of water and sanitation for all"
sgd_number = "6"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)




--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 12 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:
1. access to clean drinking water and sanitation

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:

--- Processing Page 19 ---
Extracted Arguments:

--- Processing Page 20 ---
Extracted Arguments:

--- Processing Page 21 ---
Extracted Arguments:

--- Processing Page 22 ---
Extracted Arguments:

--- Processing Page 24 ---
Extracted Arguments:

--- Processing Page 25 ---
Extracted Arguments:

--- Processing Page 26 ---
Extracted Arguments:

--- Processing Page 27 ---
Extracted Arguments:

--- Processing Page 28 ---
Extracted Arguments:

--- Processing Page 2

## SGD 7: Clean Energy

In [41]:
topic = "SGD 7 (Clean Energy): Ensure access to affordable, reliable, sustainable and modern energy for all"
sgd_number = "7"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:
1. rises in price levels and energy costs, alongside disruptions in global food markets, have adversely affected SDG targets related to poverty and inequality (SDGs 1 and 10), affordable energy (SDG 7) and food security (SDG 2).
2. Over 70% indicated an increase in electricity costs, putting the achievement of SDG 7 at risk

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:
1. increasing demand for renewable and domestic energy sources (SDG 7)

--- Processing Page 11 ---
Extracted Arguments:
1. SDG 7 Affordable and clean energy has gained importance for LRGs since the start of Russia’s war of aggression against Ukraine. Twenty-three percent of responding LRGs reported it had become their top priority, while an additional 57% stated that it had increased in relevance.
2. The growing pressures on the international energy market have led 36% of LRGs or their public energy provider to ramp up the 

## SGD 8: Decent Work, Economic Growth

In [42]:
topic = "SGD 8 (decent work, economic growth): Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all"
sgd_number = "8"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 12 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:
1. Industrial production reflects such uncertainty, with its global growth having slowed down from 6.2% in 2021 to 2.3% in 2022 as a result of inflation, energy price shocks, disruptions in supply chains for raw materials and intermediate goods, and a broader global economic deceleration.
2. Recent economic indicators indicate a slight slowdown in gross domestic product (GDP) growth, with attacks on ships in the Red Sea raising shipping costs and lengthening delivery times, disrupting production schedules and raising price pressures.

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 

## SGD 9: Infrastructure, industrilization, innovation

In [43]:
topic = "SGD 9 (Infrastructure, industrilization, innovation): Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation"
sgd_number = "9"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:
1. Moreover, in 2021, LRGs accounted for 55% of public investment in OECD countries, and, because they are typically responsible for critical areas such as water, housing, transport, infrastructure, land use and climate change, at least 105 of the 169 targets that underlie the 17 SDGs are contingent upon the active engagement of LRGs.

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 12 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:
1. Cities and regions hold a pivotal role in achieving Sustainable Development Goals (SDGs).
2. Local and regional governments (LRGs) are crucial partners of national governments in ensuring the effective, location-specific implementation of the SDGs.
3. Their activities span numerous policy areas, including but not limited to housing, transportation, infrastructure, l

## SGD 10: Inequality

In [44]:
topic = "SGD 10 (Inequality): Reduce inequality within and among countries"
sgd_number = "10"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:
1. More specifically, rises in price levels and energy costs, alongside disruptions in global food markets, have adversely affected SDG targets related to poverty and inequality (SDGs 1 and 10), affordable energy (SDG 7) and food security (SDG 2).

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:
1. In the aftermath of the COVID-19 pandemic and Russia’s war of aggression against Ukraine, the survey puts the spotlight on the declining standards of living in many cities and regions (SDGs 1 and 10).
2. Political leadership is the main success factor in targeting the SDGs at the local level, while the lack of financial resources represents a challenge.

--- Processing Page 11 ---
Extracted Arguments:
1. Most LRGs (83%) reported that living standards have worsened because of the increase in the cost of living. In response to rising prices, two-thirds of responding LRGs have implemented measures to

## SGD 11: Sustainable cities

In [45]:
topic = "SGD 11 (Sustainable Cities, Sustainable Communities): Make cities and human settlements inclusive, safe, resilient and sustainable"
sgd_number = "11"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:
1. Cities and regions play a pivotal role in steering the SDGs back on track.
2. Moreover, in 2021, LRGs accounted for 55% of public investment in OECD countries, and, because they are typically responsible for critical areas such as water, housing, transport, infrastructure, land use and climate change, at least 105 of the 169 targets that underlie the 17 SDGs are contingent upon the active engagement of LRGs.
3. It also suggests potential ways forward for local and regional governments to harness the SDGs for crafting sustainable urban and regional development policies, combating rising price levels, incentivising decarbonisation in both production and in consumption, promoting sustainable food systems and reducing food waste.
4. Both our organisations are also leading work on financing the SDGs in cities and regions, including via the SDSN Global Commission for Urban SDG Finance that which will publish its final recommendations in the 

## SGD 12: Responsible Consumption, Responsible Production

In [46]:
topic = "SGD 12 (Responsible Consumption, Responsible Production): Ensure sustainable consumption and production patterns"
sgd_number = "12"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:
1. incentivising decarbonisation in both production and in consumption

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:
1. Promoting a circular economy approach (60%) is the most common programme to reduce food waste.

--- Processing Page 12 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:
1. Their activities span numerous policy areas, including but not limited to housing, transportation, infrastructure, land use, waste management, access to clean drinking water and sanitation, energy efficiency and addressing climate change.
2. Being closest to their citizens, cities 1 Introduction energy, and improve food security.

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:

--- Pro

## SGD 13: Climate change

In [47]:
topic = "SGD 13 (Climate change): Take urgent action to combat climate change and its impacts"
sgd_number = "13"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 12 ---
Extracted Arguments:
1. Enhance public transportation options and affordability (e.g. temporary reduction in ticket prices for those most in need) to counter the financial burden of increasing fuel prices and incentivise the usage of low-carbon mobility options to help meet climate objectives.
2. Negotiate long-term power purchase agreements with energy providers to finance the installation of greener sources of energy and harness group purchasing, such as Community Choice Aggregation, to procure a less carbon-intensive mix of energy.
3. Promote the generation of local energy for local consumption, e.g. by incentivising district heating or cooling systems to foster more sustainable and energy-efficient alternatives compared to individual building-based solutions.
4. S

## SGD 14: Life bellow water

In [48]:
topic = "SGD 14 (Life bellow Water): Conserve and sustainably use the oceans, seas and marine resources for sustainable development"
sgd_number = "14"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 12 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:

--- Processing Page 19 ---
Extracted Arguments:

--- Processing Page 20 ---
Extracted Arguments:

--- Processing Page 21 ---
Extracted Arguments:

--- Processing Page 22 ---
Extracted Arguments:

--- Processing Page 24 ---
Extracted Arguments:

--- Processing Page 25 ---
Extracted Arguments:
1. The planet dimension (SDGs 6 and 12-15), which comprises the SDGs on water, sustainable consumption and production, climate action and life on land and underwater, received a slightly lower priority (20%), followed 

## SGD 15: Life on land

In [49]:
topic = "SGD 15 (Life on land): Protect, restore and promote sustainable use of terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss"
sgd_number = "15"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 12 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:

--- Processing Page 19 ---
Extracted Arguments:

--- Processing Page 20 ---
Extracted Arguments:

--- Processing Page 21 ---
Parse failed: Invalid json output: <think>
Alright, I need to figure out how to approach this task. The user has provided a specific query where they want me to extract arguments related to SDG 15 from a given text. Let's break down the problem step by step.

First, understanding what SDG 15 entails is crucial. It focuses on protecting and restoring terrestrial ecosystems, sustainabl

## SGD 16: Peace, Justice, Strong Institutions

In [50]:
topic = "SGD 16 (Peace, Justice, Strong Institutions): Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels"
sgd_number = "16"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 12 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:
1. Political leadership at the local and regional levels is the most important success factor in SDG implementation for subnational governments
2. Thirty-eight percent of respondents considered it the most important success factor for implementing the SDGs.
3. 41% of cities and regions highlighted stakeholder engagement in local and regional policy making as the most effective contribution to the SDG to strengthen citizens’ trust in their governments.

--- Processing Page 19 ---
Extracted Arguments:

--- Pr

## SGD 17: Partnerships, sustainable development

In [51]:
topic = "SGD 17 (Partnerships, sustainable development):Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development"
sgd_number = "17"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 12 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:
1. Bridging the SDG funding gap in cities

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:
1. However, only half of surveyed LRGs and territorial stakeholders contributed to their national government’s Voluntary National Reviews (VNRs), suggesting potential for improving co-operation between different levels of government.
2. 41% of cities and regions highlighted stakeholder engagement in local and regional policy making as the most effective contribution to the SDG to strengthen citizens’ trust in their governments.

--- Processing Page 19 ---
Extracted 

## SGD 0: Overarching terms

In [52]:
topic = "SGD Overarching terms: Sustainable Development Goal, SDG, Agenda 2030, leave no one behind, Voluntary National Review, SDG transformations, "
sgd_number = "0"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:
1. Only 15% of them are considered to be on track for achievement by the 2030 deadline.
2. Progress towards 48% of the SDG targets is currently insufficient, and 37% are either stagnating or regressing, including on key targets such as those related to poverty, hunger and climate action.
3. Cities and regions play a pivotal role in steering the SDGs back on track.
4. The principle of subsidiarity emphasises the importance of taking decisions at the territorial level where they will have their maximum effect.
5. Moreover, in 2021, LRGs accounted for 55% of public investment in OECD countries, and, because they are typically responsible for critical areas such as water, housing, transport, infrastructure, land use and climate change, at least 105 of the 169 targets that underlie the 17 SDGs are contingent upon the active engagement of LRGs.
6. This paper paints a comprehensive picture of how recent shocks the COVID-19 pandemic, a changing l